# Juntar archivos en uno solo
***

El objetivo de este notebook es juntar todos los netcdf de todas las secciones, y unirlas en un único archivo, de forma que se concatenan los perfiles (N_PROF), y el número de niveles es el máximo entre todos.

Importamos paquetes

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import tqdm as tqdm
import pickle
import glob

from ReadOriginalData import vars_coords_interest


Vamos a leer todos los archivos, y vamos a quedarnos con el número total de N_LEVELS y N_PROF. De forma que para cada variable tendremos un array N_PROF x N_LEVELS (En el caso del archivo con la presión interpolada, será N_PROF x max(presion_interp)).

Nos quedaremos con datos importantes como el section_id, latitud y longitud, fecha, y nombre del archivo para cada perfil, entre otros.

In [2]:
path = "./Data/corrected_sections_filtrado/"
sections = sorted(glob.glob(path+'/*'))

lats = []
lons = []
dates = []
file_name = [] # Lista con el nombre del archivo para cada PERFIL
file_names = [] # Lista con nombre del archivo para cada ARCHIVO
section_id = []
N_PROF = []

verbose = True

In [ ]:
sections

In [ ]:
for section in sections:
    sec_prof = 0
    section_path = section + "/"
    files = sorted(os.listdir(section_path))    
    for file in files:
        print(file)
        try:
            assert file.endswith(".nc")
            ds = xr.open_dataset(section_path + file)
            vars_, coords, _ = vars_coords_interest(ds)
            N_PROF.append(np.max(ds.N_PROF.values))
            file_names.append(file)
            sec_prof += np.max(ds.N_PROF.values)
            for prof in range(np.max(ds.N_PROF.values)):
                section_id.append(section)
                lats.append(ds.latitude.values[prof])
                lons.append(ds.longitude.values[prof])
                file_name.append(file)
                dates.append(np.datetime64(ds.time.values[prof]) if "time" in ds.coords else np.datetime64(file[:4]))

            press_max = np.max(ds.pressure_interp)
            ds.close()

        except AssertionError:
            continue


Definimos el N_PROF y N_LEVELS del dataset total.

In [5]:
N_PROF_total = np.sum(N_PROF)
N_LEVELS_max = int(press_max)

Vamos a hacer una concatenación en columnas de las variables en cada perfil. En este caso, como todos los archivos filtrados tienen el mismo rango en la presión, no hay que hacer padding.

El plan general es crear un diccionario que contiene la variable correspondiente para cada sección, para más tarde hacer una concatenación en la dimensión de los perfiles. Esto es más eficiente que crear directamente el array con todos los perfiles de todos los archivos, por problemas de memoria.

In [ ]:
temp_dict = {}
sal_dict = {}
ox_dict = {}
verbose = True
for i, section in enumerate(sections):
    ctd_temperature = np.array([])
    ctd_salinity = np.array([])
    ctd_oxygen = np.array([])
    section_path = section + "/"
    files = sorted(os.listdir(section_path))
    for idx, file in tqdm.tqdm(enumerate(files), total = len(files), desc = f"Section {section}") if verbose == True else enumerate(files):
            try:
                assert file.endswith(".nc")
                ds = xr.open_dataset(section_path + file)
                temp = ds["ctd_temperature_filt"].to_numpy()
                sal = ds["ctd_salinity_filt"].to_numpy()
                ox = ds["ctd_oxygen_filt"].to_numpy() if "ctd_oxygen_filt" in ds.data_vars else np.full((np.max(ds.N_PROF.values), N_LEVELS_max + 1), np.nan)
                for ip, profile in enumerate(range(np.max(ds.N_PROF.values))):
                    temp_ = temp[profile].copy()
                    sal_ = sal[profile].copy()
                    ox_ = ox[profile].copy()
                    if idx == 0 and ip == 0:
                        ctd_temperature = temp_
                        ctd_salinity = sal_
                        ctd_oxygen = ox_
                    else:
                        ctd_temperature = np.column_stack((ctd_temperature, temp_))
                        ctd_salinity = np.column_stack((ctd_salinity, sal_))
                        ctd_oxygen = np.column_stack((ctd_oxygen, ox_))
                ds.close()
            except AssertionError:
                continue

    temp_dict[section] = ctd_temperature
    sal_dict[section] = ctd_salinity
    ox_dict[section] = ctd_oxygen


Para no tener que ejecutar este código todo el rato, a parte de concatenar los arrays, los guardamos.

In [ ]:
ctd_temperature_filt = np.concatenate([temp_dict[key] for key in temp_dict.keys()], axis = 1)
#np.save("./Data/arrays/ctd_temperature_filt.npy", ctd_temperature_filt)

ctd_salinity_filt = np.concatenate([sal_dict[key] for key in sal_dict.keys()], axis = 1)
#np.save("./Data/arrays/ctd_salinity_filt.npy", ctd_salinity_filt)

ctd_oxygen_filt = np.concatenate([ox_dict[key] for key in ox_dict.keys()], axis = 1)
#np.save("./Data/arrays/ctd_oxygen_filt.npy", ctd_oxygen_filt)

Aquí se cargan.

In [ ]:
# Solo necesrio si interrumpi el proceso y grabé en formato npy
#ctd_temperature_filt = np.load("./Data/arrays/ctd_temperature_filt.npy")
#ctd_salinity_filt = np.load("./Data/arrays/ctd_salinity_filt.npy")
#ctd_oxygen_filt = np.load("./Data/arrays/ctd_oxygen_filt.npy")

Creamos el dataset con xarray para su posterior guardado como archivo **.nc**

In [9]:
filt = xr.Dataset(
    data_vars = dict(
        ctd_temperature_filt = (["N_PROF", "P"], ctd_temperature_filt.T, {
            "whp_name" : "CTDTMP",
            "whp_units" : "ITS-90",
            "standard name" : "sea_water_temperature",
            "units" : "degC",
            "reference_scale" : "ITS-90"
        }),
        ctd_salinity_filt = (["N_PROF", "P"], ctd_salinity_filt.T, {
            "whp_name" : "CTDSAL",
            "whp_units" : "PSS-78",
            "standard name" : "sea_water_practical_salinity",
            "units" : "1",
            "reference_scale" : "PSS-78"
        }),
        ctd_oxygen_filt = (["N_PROF", "P"], ctd_oxygen_filt.T, {
            "whp_name" : "CTDOXY",
            "whp_units" : "UMOL/KG",
            "standard name" : "moles_of_oxygen_per_unit_mass_in_sea_water",
            "units" : "umol/kg"
        })),
    coords = dict(
        time = (["N_PROF"], dates),
        section_id = (["N_PROF"], section_id),
        file_name = (["N_PROF"], file_name),
        latitude = (["N_PROF"], lats),
        longitude = (["N_PROF"], lons)        ,
        pressure = (["P"], np.arange(0, N_LEVELS_max + 1, 1))
    )
)

filt.to_netcdf("./Data/join/total_filt.nc")